# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook demonstrates how to explore the FAIR² dataset, titled *Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya*, using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is defined by a Croissant schema accessible at:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Do NOT treat as dict or iterate over metadata

print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Explore available record sets, their field `@id`s, and a preview of fields and columns.

We inspect dataset metadata to identify `@id`s for record sets and their fields.

In [ ]:
record_sets = list(dataset.record_sets())

if len(record_sets) == 0:
    print("No record sets found in the dataset metadata. Listing possible encodings or distributions...")
    if hasattr(metadata, 'distribution'):
        for i, dist in enumerate(metadata.distribution):
            print(f"Distribution {i} @id: {getattr(dist, '@id', '[no @id]')}")
    else:
        print("No 'distribution' found in metadata either.")
else:
    print("Found record sets:")
    for rs in record_sets:
        print(f"  - @id: {rs['@id']}")
        fields = rs.get('field') or rs.get('fields')
        if fields:
            print("    Fields/Columns:")
            for field in fields:
                print(f"      - @id: {field['@id']} | name: {field.get('name', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
We'll attempt to extract all record sets. If none are defined, we'll explore possible DataFrame loading from available distributions or fall back to inspecting raw records.

In [ ]:
# Attempt to extract records for all record sets
dataframes = {}

record_sets = list(dataset.record_sets())
# Use the @id of each record set for referencing
record_set_ids = [rs['@id'] for rs in record_sets]

for record_set_id in record_set_ids:
    # Fetch records as list of dicts
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for record set '{record_set_id}' with shape {dataframes[record_set_id].shape}")
    else:
        print(f"No records found for record set {record_set_id}")

if len(dataframes) > 0:
    # Pick the first available DataFrame to show columns and preview
    first_record_set = list(dataframes.keys())[0]
    print(f"\nColumns of first record set [{first_record_set}]:")
    print(dataframes[first_record_set].columns.tolist())
    dataframes[first_record_set].head()
else:
    print("No tabular dataframes could be loaded from the record sets.\nTry loading distributions directly (if supported by mlcroissant version).")

## 4. Exploratory Data Analysis (EDA)
Let's perform typical EDA operations using the DataFrame.
- Filter records based on a numeric field.
- Normalize that field for filtered records.
- Group or aggregate by a chosen column (such as a categorical field, if available).

*Note: All column and field references use their `@id`.*

In [ ]:
# If a DataFrame is available, proceed with EDA
if len(dataframes) > 0:
    # Choose the first DataFrame and display its columns
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Available columns for record set {record_set_id}:\n", df.columns.tolist())

    # Try to infer likely numeric fields: look for columns with 'coef', 'std', 'pval', 'log', 'iteration', etc.
    numeric_field_candidates = [c for c in df.columns if any(s in c.lower() for s in ['coef', 'std', 'value', 'log', 'iteration'])]
    if len(numeric_field_candidates) == 0:
        print("No obvious numeric fields found for demonstration.")
        numeric_field = df.columns[0] if len(df.columns)>0 else None
    else:
        numeric_field = numeric_field_candidates[0]
    print(f"Using '{numeric_field}' as an example numeric field.")

    # Choose a threshold (e.g., above median if >10 unique, else arbitrary)
    if numeric_field is not None and pd.api.types.is_numeric_dtype(df[numeric_field]):
        threshold = df[numeric_field].median()
    else:
        threshold = 10

    filtered_df = df[df[numeric_field] > threshold] if numeric_field is not None else df.copy()
    print(f"Filtered records where '{numeric_field}' > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field (z-score)
    if numeric_field is not None and pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field (e.g., 'variable', 'category', 'predictor', etc.)
    group_field_candidates = [c for c in df.columns if any(s in c.lower() for s in ['variable', 'group', 'category', 'predictor'])]
    if len(group_field_candidates) > 0:
        group_field = group_field_candidates[0]
        print(f"\nGrouping filtered data by '{group_field}':")
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(grouped_df.head())
    else:
        print("No appropriate group field found for grouping.")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Let's visualize the distribution of the numeric field chosen above, and examine relationships if grouping is possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0:
    # Use prior variables
    df = dataframes[record_set_id]
    # Numeric field from EDA block
    if numeric_field is not None and numeric_field in df:
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field].dropna(), kde=True)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel("Frequency")
        plt.show()

    if 'group_field' in locals() and group_field is not None and group_field in df:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No DataFrame available for visualization.")

## 6. Conclusion
In this notebook, we've demonstrated loading and exploring the FAIR² dataset using `mlcroissant`.

- **Metadata** was accessed programmatically to retrieve the dataset's structure and descriptive details.
- **Record sets** and **fields** were examined via their `@id`s, with data extraction into Pandas DataFrames.
- We performed **filtering**, **normalization**, and **grouping** of a key numeric field, then visualized results.
- This approach enables reproducible, schema-driven data exploration for complex, multi-part datasets as defined in Croissant.

*For further analysis, consult the dataset schema for the full set of record sets, fields, and their semantic definitions via their `@id`s.*